In [13]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [14]:
# Define image transform
transform = transforms.Compose([
    transforms.Resize((64, 64)),           # Resize images to 64x64
    transforms.ToTensor(),                 # Convert to tensor
    transforms.Normalize([0.5]*3, [0.5]*3) # Normalize to [-1, 1]
])

# Load dataset
dataset = datasets.ImageFolder(root='data', transform=transform)

# Create DataLoader
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

# Example: fetch a batch
for images, _ in dataloader:
    print("Batch shape:", images.shape)  # Should be (32, 3, 64, 64)
    break

Batch shape: torch.Size([1, 3, 64, 64])


In [20]:
import torch
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)
    
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
        
    def forward(self, x):
        return self.model(x)

# Training setup
lr = 0.001
num_epochs = 30
loss_function = nn.BCELoss()
batch_size = 32

generator = Generator()
discriminator = Discriminator()

optimizer_discriminator = torch.optim.Adam(discriminator.parameters(), lr=lr)
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=lr)


In [23]:
for epoch in range(num_epochs):
    for n, (real_samples, _) in enumerate(dataloader):
        current_batch_size = real_samples.size(0)

        # Data for training the discriminator
        real_samples_labels = torch.ones((current_batch_size, 1))
        latent_space_samples = torch.randn((current_batch_size, 2))
        generated_samples = generator(latent_space_samples)
        generated_samples_labels = torch.zeros((current_batch_size, 1))

        all_samples = torch.cat((real_samples, generated_samples), dim=0)
        all_labels = torch.cat((real_samples_labels, generated_samples_labels), dim=0)

        # Train discriminator
        discriminator.zero_grad()
        output_discriminator = discriminator(all_samples)
        loss_discriminator = loss_function(output_discriminator, all_labels)
        loss_discriminator.backward()
        optimizer_discriminator.step()

        # Train generator
        latent_space_samples = torch.randn((current_batch_size, 2))
        generator.zero_grad()
        generated_samples = generator(latent_space_samples)
        output_discriminator_generated = discriminator(generated_samples)
        loss_generator = loss_function(output_discriminator_generated, real_samples_labels)
        loss_generator.backward()
        optimizer_generator.step()

        # Show loss
        if epoch % 10 == 0 and n == len(dataloader) - 1:
            print(f"Epoch {epoch} / {num_epochs}, Batch {n} / {len(dataloader)}, "
                  f"Discriminator Loss: {loss_discriminator.item()}, "
                  f"Generator Loss: {loss_generator.item()}")


RuntimeError: Tensors must have same number of dimensions: got 4 and 2